# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("Date Published:", metadata.datePublished)


## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields for clarity and reproducibility.

In [ ]:
# Access record sets
record_sets = dataset.record_sets
print(f"Total record sets found: {len(record_sets)}")

# Display @id and name for each record set
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}, Name: {rs.get('name', '<Unnamed>')}")
    # Display fields within each record set
    fields = rs.get('fields', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    Field @id: {field['@id']}, Name: {field.get('name', '<Unnamed>')} (Type: {field.get('dataType', '<unknown>')})")
    else:
        print("  No fields available.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview for reproducibility.

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    # Extract records for each record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set {rs_id} with shape {df.shape}")
        print("Columns:", df.columns.tolist())
        print(df.head(3))
    else:
        print(f"No records found for record set {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All operations reference fields by their `@id`.

In [ ]:
# Pick a record set for analysis (use the first available one)
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes.get(record_set_id)

    # List available numeric fields (fields with Integer or Float type)
    fields_info = next((rs for rs in record_sets if rs['@id'] == record_set_id), {})
    numeric_fields = [
        field['@id'] for field in fields_info.get('fields', [])
        if field.get('dataType', '').lower() in ('integer', 'float', 'number')
    ]
    print("Numeric field @ids:", numeric_fields)

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = 10

        # Filter DataFrame using the numeric field
        try:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())

            # Normalize the numeric field
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Try to group by a categorical field
            group_fields = [
                field['@id'] for field in fields_info.get('fields', [])
                if field.get('dataType', '').lower() == 'text' and field['@id'] != numeric_field_id
            ]
            if group_fields:
                group_field_id = group_fields[0]
                if group_field_id in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                    print(f"Grouped average {numeric_field_id} by {group_field_id}:")
                    print(grouped_df.head())
                else:
                    print(f"Group field {group_field_id} not found in DataFrame columns.")
            else:
                print("No suitable group fields found.")
        except Exception as e:
            print(f"EDA could not be performed due to: {e}")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing fields by their `@id` where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if EDA produced a filtered DataFrame
if 'filtered_df' in locals() and not filtered_df.empty and numeric_fields:
    numeric_field_id = numeric_fields[0]
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=10)
    plt.title(f"Distribution of {numeric_field_id} (> threshold)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field_id' in locals() and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric fields or filtered DataFrame available for visualization.")

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR^2 dataset using the `mlcroissant` library. By referencing all record sets and fields via their `@id`, the workflow ensures reproducibility and consistency. The available tabular record sets were examined, basic EDA was performed on numeric fields, and key distributions were visualized. For deeper analysis, consult the dataset's Croissant schema documentation and consider adding more domain-specific statistical steps.